# 进阶实践项目 01：MRI 肿瘤分割中的边界、鲁棒性与不确定性

MRI 分割模型不仅需要给出病灶区域，还要面对边界模糊、小病灶、空切片、扫描差异和过度自信等问题。本项目研究一个具体问题：在患者级独立评价下，怎样判断模型的区域重叠、边界质量和预测稳定性。


> **实践定位**
>
> 这不是短时间代码竞赛，也不是以最高分数决定完成度的作业。可以只完成数据核对、基线、一个消融实验或一段严谨的失败分析。问题定义、文献依据、方法选择、验证设计、错误解释和下一步实验，重要性高于单一性能数值。
>
> 最终提交由两部分组成：当前 Notebook，以及一份设计报告。设计报告不是代码说明书，而是研究方案说明。参考答案只展示一种能够运行的方案，不代表唯一正确答案，也不意味着其中的模型一定最适合你的目标。


### 可完成的最低范围

完成数据配对与患者级划分，训练一个轻量分割基线，报告 Dice 和至少一种补充指标，并解释两例成功或失败病例。模型未训练完成时，也可以提交数据审计、损失函数比较设计和预期错误分析。

### 可选扩展

残差 U-Net、Tversky/Focal-Tversky、边界损失、深监督、测试时增强、MC dropout、小型集成、按病灶大小分层评价。


## 主题背景

病灶分割把每个像素或体素标记为目标或背景。Dice 适合衡量整体重叠，但两个具有相近 Dice 的结果可能具有完全不同的边界误差：一个整体平移，另一个只漏掉细小突起。临床或科研使用还可能关心病灶体积、漏检、误检、边界距离和人工修改成本。

不确定性估计用于识别模型自身不稳定的区域。测试时增强、MC dropout 和模型集成都可以产生多次预测，再观察预测之间的差异。差异较大说明模型对扰动或参数变化敏感，但它不自动等于真实错误概率，也不保证已经校准。


## 数据来源与 Kaggle 获取

**推荐数据：体验项目 01 使用的 LGG MRI 数据。** 在 Kaggle Notebook 右侧选择 **Add Data**，搜索 `Brain MRI segmentation`，或直接添加数据集 `mateuszbuda/lgg-mri-segmentation`。数据包含 110 名 TCGA 低级别胶质瘤患者的 FLAIR 图像和人工异常区域掩膜，常见文件结构是患者文件夹内的 `.tif` 图像及同名 `_mask.tif` 掩膜。

- Kaggle 数据页：https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation
- 数据来源说明：Kaggle 页面注明影像来自 TCIA 的 TCGA-LGG 集合。

已经完成体验项目时，可以直接复用同一 Kaggle 数据挂载，不需要再次下载。更大规模的 BraTS 数据可作为拓展，但数据体积、三维预处理和多类别标签都会显著增加难度，不是本项目的必要条件。

运行下一格后应看到真实数据文件数、患者数和若干配对样例。若没有检测到数据，不要继续训练，先用 AI/Agent 根据实际 `/kaggle/input` 文件树修正路径。


In [ ]:
from pathlib import Path
import re, pandas as pd
ROOT=Path('/kaggle/input')
files=list(ROOT.glob('**/*')) if ROOT.exists() else []
masks=[p for p in files if p.is_file() and '_mask' in p.stem.lower() and p.suffix.lower() in {'.tif','.tiff','.png'}]
rows=[]
for m in masks:
    image=Path(str(m).replace('_mask',''))
    patient=m.parent.name
    rows.append({'patient':patient,'image':str(image),'mask':str(m),'paired':image.exists()})
pairs=pd.DataFrame(rows)
print('mask files:',len(masks),'paired:',int(pairs.paired.sum()) if len(pairs) else 0)
print('patients:',pairs.loc[pairs.paired,'patient'].nunique() if len(pairs) else 0)
display(pairs.head())


## AI 与 Agent 的使用

可以使用 ChatGPT、代码 Agent、Kaggle Notebook Assistant 或其他工具完成资料检索、数据目录检查、代码解释、报错定位、方法比较和报告整理。建议把 AI 当作可审查的协作者，而不是答案来源。

适合交给 AI/Agent 的工作包括：

- 根据实际文件树改写数据读取函数；
- 解释一段代码的输入、输出、shape 和潜在泄漏；
- 比较两种损失、模型或指标的适用条件；
- 根据报错和当前变量状态提出最小修改；
- 搜索论文后整理研究问题、数据、方法、评价和局限；
- 把实验日志整理成设计报告草稿。

所有生成内容都需要核对。论文标题和链接必须打开确认；代码必须逐格运行；数据划分必须用实际 ID 检查；任何“性能提升”都必须由同一测试条件下的结果支持。建议在设计报告末尾记录主要提示词、接受了哪些建议、拒绝了哪些建议以及原因。


## 文献调研任务

至少阅读两项资料，并在设计报告中比较它们的研究问题与方法，而不是只摘录摘要。

- U-Net：编码器获取上下文，解码器恢复定位，跳跃连接保留高分辨率信息。https://doi.org/10.1007/978-3-319-24574-4_28
- LGG 数据相关研究：Buda 等使用深度学习提取低级别胶质瘤形状特征。数据页给出论文信息。https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation
- Tversky loss：通过调整假阳性和假阴性的权重处理病灶比例不平衡。https://arxiv.org/abs/1706.05721
- Kendall 与 Gal：区分观测噪声与模型不确定性。https://papers.nips.cc/paper/2017/hash/2650d6089a6d640c5e85b2b88265dc2b-Abstract.html

调研时记录：论文使用二维还是三维输入，独立样本单位是什么，损失函数试图改善哪类错误，评价是否包含边界或逐病例结果。


## 任务 1：确定研究重点

选择一个主要问题，并写出可以被实验检验的假设。

- 小病灶漏检：Tversky 或 Focal-Tversky 是否提高召回；
- 边界偏差：边界相关损失是否降低 HD95；
- 输入扰动敏感：测试时增强差异是否集中在错误区域；
- 模型复杂度：残差块是否在相同训练轮次下更稳定。

不要同时堆叠所有改动。一个清楚的单因素比较通常比多个改动混在一起更容易解释。


## 任务 2：患者级划分与数据检查

同一患者包含多张相邻切片。随机按切片划分会让高度相似的图像进入训练和测试。请完成患者 ID 提取、训练/验证/测试患者集合检查，并绘制至少 8 组图像与掩膜。


In [ ]:
# TODO 1：只保留 paired=True 的记录。
# TODO 2：按 patient 分组划分，禁止同一 patient 跨集合。
# 可选工具：GroupShuffleSplit 或 StratifiedGroupKFold。
# TODO 3：打印三个集合的患者 ID 交集，应全部为空。

# 示例断言：
# assert set(train_patients).isdisjoint(test_patients)


## 任务 3：选择模型与损失

轻量 U-Net 已经足以形成有效基线。模型选择应由问题驱动：残差连接用于改善优化，注意力用于重新分配特征，三维卷积用于利用层间连续性，但都会增加计算和调试成本。

损失函数可以从以下方案选择：

- `BCE + Dice`：稳定、容易实现，适合作为主基线；
- `Tversky`：可以提高对假阴性的关注，适合前景很小且漏检代价较高的情形；
- `Focal-Tversky`：进一步强调困难像素，但参数更多；
- 边界损失：直接利用距离或表面信息，适合边界问题，但需要更谨慎的实现。


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO：实现轻量 U-Net、残差 U-Net 或你调研后选择的结构。
        pass
    def forward(self,x):
        raise NotImplementedError

def loss_fn(logits, mask):
    # TODO：实现并解释 BCE-Dice、Tversky 或其他损失。
    raise NotImplementedError


## 任务 4：评价、病例分析与不确定性

至少报告 Dice 和一种互补评价。IoU 与 Dice 都描述重叠，信息高度相关；HD95、表面 Dice、体积误差、病灶检出率或按病灶大小分层更能补充不同错误。HD95 若没有体素间距，只能报告为像素距离。

不确定性可选择测试时增强、MC dropout 或小型集成。请展示概率图、不确定性图和至少两例错误。需要解释不确定性高但预测正确、或不确定性低但预测错误的可能原因。


In [ ]:
# TODO：训练与评价。
# 建议输出：
# 1) 训练/验证曲线；
# 2) MRI、真实掩膜、预测掩膜、误差图；
# 3) 不确定性或多次预测差异；
# 4) 按患者汇总的指标表，而不是只按切片平均。


## 设计报告是主要提交内容

报告应能够让没有运行 Notebook 的读者理解你的问题、选择和证据。建议正文包含以下内容：

1. **研究问题与动机**：具体要解决什么问题，为什么值得研究，输出将被怎样使用；
2. **数据来源与适用范围**：数据来自体验项目、Kaggle、UCI 或其他公开来源，样本单位、标签、许可、已知偏差和不能代表的人群；
3. **文献调研**：至少阅读两篇原始论文或官方方法文档，说明它们解决的问题、关键方法、评价方式和可借鉴之处；
4. **方案候选与选择理由**：列出考虑过的模型、损失、特征或指标，说明最终选择与算力、样本量、目标和风险之间的关系；
5. **数据划分与验证**：独立样本是谁，怎样避免同一患者、玻片或空间邻域跨集合，哪些指标对应哪些错误；
6. **实现进度与证据**：已经运行的代码、图表、失败现象、异常样本和未完成部分；
7. **结果解释**：结果支持什么、不支持什么，性能较低或没有训练完成也要解释原因；
8. **局限与下一步**：最可能改变结论的限制，以及下一项最值得做的实验；
9. **AI/Agent 使用记录**：主要提示词、采用的建议、人工核查方式和仍未解决的问题。

报告评价重点是思路是否清楚、选择是否有依据、验证是否可信、解释是否诚实。准确率、Dice、AUC 或相关系数只是一部分证据。


### 项目 01 报告还需要回答

- 分割标签具体表示什么，空掩膜切片怎样处理；
- 选择的损失更关注假阳性还是假阴性；
- Dice 与边界指标出现冲突时怎样解释；
- 你的不确定性量度反映哪一种变化，为什么不能直接当作错误概率；
- 模型用于新中心或不同 MRI 参数时最可能失败在哪里。
